# Exact-cylinder steady Stokes

Author the geometry in Python, compile the shipped Eqiora equations, resolve one numerical plan, and inspect its common Result.

In [ ]:
from importlib.resources import files

import eqiora
import eqiora.matplotlib as eqplot

In [ ]:
geometry_graph = eqiora.geometry.GeometryGraph()
rectangle = geometry_graph.rectangle(x_bounds=(0.0, 2.2), y_bounds=(0.0, 0.41))
circle = geometry_graph.circle(center=(0.2, 0.2), radius=0.05)
fluid = geometry_graph.subtract(rectangle, circle)
geometry = geometry_graph.build(
    fluid,
    named_topology={
        "fluid": fluid.region,
        "inlet": rectangle.boundaries[0],
        "outlet": rectangle.boundaries[1],
        "walls": rectangle.boundaries[2:4],
        "cylinder": circle.boundaries[0],
    },
)
geometry

In [ ]:
mesh_request = eqiora.meshing.GmshMesher(
    maximum_boundary_error=1e-4,
    minimum_mean_ratio=1e-5,
    maximum_boundary_facets=50,
)
mesh_plan = eqiora.meshing.resolve(geometry, mesh_request)
mesh = eqiora.meshing.generate(geometry, plan=mesh_plan)
mesh

In [ ]:
source_path = files(eqiora).joinpath("examples", "steady-flow-past-cylinder.eqi")
model = eqiora.compile(
    path=source_path,
    geometry=geometry,
    parameters={
        "dynamic_viscosity": 1.0e-3,
        "zero_pressure": 0.0,
        "inlet_speed": 0.3,
        "channel_height": geometry.bounds[1][1] - geometry.bounds[1][0],
    },
)
model

In [ ]:
linear = eqiora.solve.Linear(
    relative_tolerance=1e-6,
    absolute_tolerance=1e-13,
    maximum_iterations=10_000,
)
stokes_plan = eqiora.resolve(
    model,
    mesh=mesh,
    spatial=eqiora.fem.MiniP1(),
    solve=linear,
    scaling=None,
)
stokes_plan

In [ ]:
result = eqiora.run(stokes_plan)
result

In [ ]:
pressure = result.output(stokes_plan.capability.pressure)
assert result.plan_key == stokes_plan.identity
summary = {
    "geometry": geometry.digest,
    "mesh": mesh.digest,
    "model": model.digest,
    "plan": stokes_plan.identity,
    "result": result.plan_key,
    "pressure_vertices": pressure.coefficient_count("vertex"),
}
summary

In [ ]:
pressure_figure = eqplot.plot_scalar_field(result, field=stokes_plan.capability.pressure)
pressure_figure